In [1]:
import os
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import networkx as nx

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

input_dir = "/home/ajarrah/PhD_Thesis/gene_paper/results_halfbrain"
output_dir = "/home/ajarrah/PhD_Thesis/gene_paper/GSEA_results_halfbrain_adj_p_val_pct5"

os.makedirs(output_dir, exist_ok=True)

# Configurations

In [2]:
human = False
adj_pval = True
cut_off_dotplot = 0.05


# Files

In [3]:
files = [
    "DE_AAD_vs_AC_Aged_AD_vs_Aged_Control.csv",
    "DE_AAD_vs_YAD_Aged_AD_vs_Young_AD.csv",
    "DE_AC_vs_YC_Aged_Control_vs_Young_Control_(aging_effect).csv",
    "DE_AD_vs_Control_All_AD_vs_All_Control.csv",
    "DE_Aged_vs_Young_All_Aged_vs_All_Young.csv",
    "DE_YAD_vs_YC_Young_AD_vs_Young_Control.csv"
]

# Pathway databases

In [4]:
gene_sets = {

    # Broad biological programs
    "Hallmark": {
        "library": "MSigDB_Hallmark_2020",
        "min_size": 10,
        "max_size": 500,
    },

    # Cellular processes
    "GO_BP": {
        "library": "GO_Biological_Process_2023",
        "min_size": 15,
        "max_size": 1000,
    },

    # Curated signaling pathways
    "Reactome": {
        "library": "Reactome_2022",
        "min_size": 10,
        "max_size": 500,
    },

    # Metabolic/signaling pathways
    "KEGG": {
        "library": "KEGG_2019_Mouse",
        "min_size": 10,
        "max_size": 300,
    },

    # Brain-related pathways
    "WikiPathways": {
        "library": "WikiPathways_2024_Mouse",
        "min_size": 10,
        "max_size": 500,
    },

    # Disease-associated genes
    "DisGeNET": {
        "library": "DisGeNET",
        "min_size": 10,
        "max_size": 500,
    }
}

# Create ranking

In [5]:
def create_rank_file(df):

    df = df.copy()

    # remove missing values
    df = df.dropna(subset=[
        "gene",
        "log2FC",
        "padj",
        "pval"
    ])

    # remove duplicated genes
    df = df.drop_duplicates( subset="gene", keep="first")

    # avoid log(0)
    df["padj"] = df["padj"].clip(lower=1e-300)
    df["pval"] = df["pval"].clip(lower=1e-300)

    # GSEA ranking metric
    #I used pval instead of padj because padj is too conservative and may lead to missing important genes
    if adj_pval:
        df["rank"] = ( np.sign(df["log2FC"]) * -np.log10(df["padj"])) 
    else:
        df["rank"] = ( np.sign(df["log2FC"]) * -np.log10(df["pval"]))  
    ranking = (df[["gene","rank"]].sort_values("rank", ascending=False ))

    return ranking

# Cnet plot function

In [6]:
def make_cnetplot(gsea_result, output):

    res = gsea_result.copy()

    # Significant pathways
    res = res[ res["FDR q-val"] < 0.05]

    if len(res) == 0:
        return

    # top pathways by NES magnitude
    res["absNES"] = abs(res["NES"])

    pathways = res.sort_values("absNES", ascending=False)
    G = nx.Graph()
    for _, row in pathways.iterrows():
        pathway = row["Term"]

        # leading edge genes
        genes = row["Lead_genes"]

        if pd.isna(genes):
            continue

        genes = genes.split(";")
        G.add_node(pathway, type="pathway")

        for gene in genes:
            G.add_node(gene, type="gene")
            G.add_edge(pathway, gene)

    if len(G.nodes)==0:
        return

    plt.figure(figsize=(12,10))

    pos = nx.spring_layout(G, seed=42 )

    pathway_nodes = [
        n for n,d in G.nodes(data=True)
        if d["type"]=="pathway"
    ]

    gene_nodes = [
        n for n,d in G.nodes(data=True)
        if d["type"]=="gene"
    ]

    nx.draw_networkx_nodes(G, pos, nodelist=pathway_nodes, node_size=1500)
    nx.draw_networkx_nodes(G, pos, nodelist=gene_nodes, node_size=300)
    nx.draw_networkx_edges(G, pos, alpha=0.4)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(output, dpi=300, bbox_inches="tight")

    plt.close()

# Run GSEA

In [ ]:
for file in files:

    print("\nRunning:", file)
    path = os.path.join( input_dir, file)

    # read DE results
    de = pd.read_csv(path)
    ranking = create_rank_file(de)
    comparison = (file.replace(".csv",""))
    rank_file = os.path.join(output_dir, comparison+"_ranking.rnk")
    ranking.to_csv(rank_file, sep="\t", index=False, header=False)

    for db_name, db in gene_sets.items():
        print("  ", db_name)
        outdir = os.path.join(output_dir, comparison, db_name)
        os.makedirs(outdir, exist_ok=True)

        if human:
            ranking["gene"] = ranking["gene"].str.upper()       # convert to human-style

        try:
            prerank = gp.prerank(
                rnk=ranking,
                gene_sets=db["library"],
                threads=4,
                permutation_num=1000,
                min_size=db["min_size"],
                max_size=db["max_size"],
                outdir=outdir,
                seed=42,
                verbose=False
            )
            results = prerank.res2d

            results.to_csv(os.path.join(outdir, "GSEA_results.csv"))

            # ----------------------------
            # CNET plot
            # ----------------------------
            
            cnet_file = os.path.join(outdir, "cnetplot.png")
            make_cnetplot(results, cnet_file)

            # ----------------------------
            # GSEA dotplot
            # ----------------------------

            gp.dotplot(
                results,
                column="FDR q-val",
                title=f"{comparison} {db_name}",
                cutoff=cut_off_dotplot,
                size=10,
                figsize=(8,6),
                ofname=os.path.join(outdir, "dotplot.png")
            )

        except Exception as e:
            print("FAILED:", db_name, e)


print("\nFinished")

/tmp/ipykernel_2526379/271232763.py:22: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-17 10:28:16,505 [WARNING] Duplicated values found in preranked stats: 43.29% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
/tmp/ipykernel_2526379/271232763.py:22: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-17 10:28:16,691 [WARNING] Duplicated values found in preranked stats: 43.29% of genes
The order of those genes will be arbitrary, which may produce unexpected results.



Running: DE_AAD_vs_AC_Aged_AD_vs_Aged_Control.csv
   Hallmark
FAILED: Hallmark Warning: No enrich terms when cutoff = 0.05
   GO_BP


/tmp/ipykernel_2526379/271232763.py:22: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-17 10:28:17,720 [WARNING] Duplicated values found in preranked stats: 43.29% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


FAILED: GO_BP Warning: No enrich terms when cutoff = 0.05
   Reactome


/tmp/ipykernel_2526379/271232763.py:22: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-17 10:28:18,095 [WARNING] Duplicated values found in preranked stats: 43.29% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


...

FAILED: Reactome Warning: No enrich terms when cutoff = 0.05
   KEGG


/tmp/ipykernel_2526379/271232763.py:22: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-17 10:28:40,332 [WARNING] Duplicated values found in preranked stats: 54.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


FAILED: KEGG Warning: No enrich terms when cutoff = 0.05
   WikiPathways


/tmp/ipykernel_2526379/271232763.py:22: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-17 10:28:41,680 [WARNING] Duplicated values found in preranked stats: 54.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-07-17 10:28:41,839 [ERROR] No gene sets passed through filtering condition !!! 
Hint 1: Try to lower min_size or increase max_size !
Hint 2: Check gene symbols are identifiable to your gmt input.
Hint 3: Gene symbols curated in Enrichr web services are all upcases.

2026-07-17 10:28:41,841 [ERROR] The first entry of your gene_sets (gmt) look like this : { 46, XX Testicular Disorders of Sex Development: [disease, NR0B1, AR, FOXL2, NR5A1, SOX3, SOX9, SOX10, SRY, RSPO1]}
2026-07-17 10:28:41,841 [ERROR] The first 5 genes look like this : [ Capn3, Bdnf, S100a10, Rtn1, Itpka ]


FAILED: WikiPathways Warning: No enrich terms when cutoff = 0.05
   DisGeNET
FAILED: DisGeNET No gene sets passed through filtering condition !!! 
Hint 1: Try to lower min_size or increase max_size !
Hint 2: Check gene symbols are identifiable to your gmt input.
Hint 3: Gene symbols curated in Enrichr web services are all upcases.


Finished
